# Phase 1 — Data

We need two instruction sets:
- **Harmful**: instructions the model should refuse (AdvBench, same as the paper)
- **Harmless**: instructions the model should comply with (Alpaca subset)

Split each into train (small, used to compute the mean-diff direction) and val/test (held out, used to measure jailbreak success and pick the best layer).

Paper's split sizes: they use small train sets (~32 examples) since the direction is a simple mean difference and doesn't need much data. We'll mirror that.

In [ ]:
!pip install -q datasets

## 1.1 — Load AdvBench (harmful instructions)

AdvBench is the standard harmful-instructions benchmark (from the GCG/llm-attacks paper, reused by Arditi et al.). It's hosted as a CSV in the llm-attacks repo.

In [ ]:
import pandas as pd

advbench_url = "https://raw.githubusercontent.com/llm-attacks/llm-attacks/main/data/advbench/harmful_behaviors.csv"
advbench_df = pd.read_csv(advbench_url)
print(advbench_df.shape)
advbench_df.head()

In [ ]:
harmful_instructions = advbench_df["goal"].tolist()
print("Total harmful instructions:", len(harmful_instructions))
print("Example:", harmful_instructions[0])

## 1.2 — Load Alpaca (harmless instructions)

We only want single-turn instructions with no extra `input` field, to keep the prompt format simple and comparable to the harmful set.

In [ ]:
from datasets import load_dataset

alpaca = load_dataset("tatsu-lab/alpaca", split="train")
alpaca_no_input = alpaca.filter(lambda x: x["input"].strip() == "")
print("Alpaca examples with no extra input:", len(alpaca_no_input))

harmless_instructions = alpaca_no_input["instruction"]
print("Example:", harmless_instructions[0])

## 1.3 — Shuffle and split

Train set: used only to compute the diff-of-means direction (small, per the paper).
Val set: used to pick the best layer/direction.
Test set: held out entirely until final evaluation, so our layer-selection isn't overfit to the same data we report results on.

In [ ]:
import random

random.seed(42)

def split_list(lst, n_train, n_val, n_test):
    lst = list(lst)
    random.shuffle(lst)
    train = lst[:n_train]
    val = lst[n_train:n_train + n_val]
    test = lst[n_train + n_val: n_train + n_val + n_test]
    return train, val, test

N_TRAIN = 32
N_VAL = 32
N_TEST = 100

harmful_train, harmful_val, harmful_test = split_list(harmful_instructions, N_TRAIN, N_VAL, N_TEST)
harmless_train, harmless_val, harmless_test = split_list(harmless_instructions, N_TRAIN, N_VAL, N_TEST)

print("Harmful  -> train:", len(harmful_train), "val:", len(harmful_val), "test:", len(harmful_test))
print("Harmless -> train:", len(harmless_train), "val:", len(harmless_val), "test:", len(harmless_test))

## 1.4 — Format with chat template

Every instruction needs to go through Qwen3's chat template before tokenizing, matching how it's actually used. This must be consistent with whatever we confirmed in Phase 0 (e.g. `enable_thinking=False` if needed).

In [ ]:
def format_prompt(instruction, tokenizer, enable_thinking=False):
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": instruction}],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=enable_thinking,
    )

# Sanity check on one example
print(format_prompt(harmful_train[0], hf_tokenizer))

## 1.5 — Save all splits to disk

So Phase 2 can load these directly without re-running the download/split logic.

In [ ]:
import json
import os

os.makedirs("data", exist_ok=True)

splits = {
    "harmful_train": harmful_train,
    "harmful_val": harmful_val,
    "harmful_test": harmful_test,
    "harmless_train": harmless_train,
    "harmless_val": harmless_val,
    "harmless_test": harmless_test,
}

for name, data in splits.items():
    with open(f"data/{name}.json", "w") as f:
        json.dump(data, f, indent=2)

print("Saved:", os.listdir("data"))